In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Math

# ==============================================================================
# PROBLEM: Pole-Zero Plots & ROC Determination for Multiple Z-Transforms
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Solution Overview (Pole-Zero Maps & ROC Analysis)</b><br>
* <b>X₁(z):</b> Pole at z = -2, Zero at z = 0.5.<br>
* <b>X₂(z):</b> Causal signal -> ROC is exterior to the outermost pole.<br>
* <b>X₃(z):</b> Absolutely summable signal -> ROC includes the unit circle.<br>
* <b>Note:</b> Use the dropdown menu below to select and visualize each transformation interactively.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

def plot_z_transforms(system_choice):
    with out:
        clear_output(wait=True)
        
        fig, ax_pz = plt.subplots(figsize=(8, 6))
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-3.0, 3.0)
        ax_pz.set_ylim(-3.0, 3.0)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        theta = np.linspace(0, 2*np.pi, 200)
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label='Unit Circle')

        x_vals = np.linspace(-3.5, 3.5, 400)
        y_vals = np.linspace(-3.5, 3.5, 400)
        X, Y = np.meshgrid(x_vals, y_vals)
        Z_dist = np.sqrt(X**2 + Y**2)

        if system_choice == "X1(z) = (1 - 0.5z^-1) / (1 + 2z^-1)":
            # X1: Zero at 0.5, Pole at -2
            zeros_x, zeros_y = [0.5], [0]
            poles_x, poles_y = [-2.0], [0]
            # Assume causal (default if not specified) -> ROC: |z| > |-2| = 2
            roc_mask = Z_dist > 2.0
            ax_pz.imshow(roc_mask, extent=(-3.5, 3.5, -3.5, 3.5), origin='lower', cmap='Greens', alpha=0.25, zorder=0)
            ax_pz.plot(2.0 * np.cos(theta), 2.0 * np.sin(theta), 'g:', linewidth=2, label='ROC Boundary (|z| = 2)')
            title_str = "X₁(z): Pole at z = -2, Zero at z = 0.5 (Causal ROC: |z| > 2)"

        elif system_choice == "X2(z) = [1 - (1/3)z^-1] / [(1 + 0.5z^-1)(1 - (2/3)z^-1)]":
            # X2: Poles at -0.5 and 2/3 ≈ 0.667, Causal -> ROC: |z| > 2/3
            zeros_x, zeros_y = [1/3], [0]
            poles_x, poles_y = [-0.5, 2/3], [0, 0]
            roc_mask = Z_dist > (2/3)
            ax_pz.imshow(roc_mask, extent=(-3.5, 3.5, -3.5, 3.5), origin='lower', cmap='Greens', alpha=0.25, zorder=0)
            ax_pz.plot((2/3) * np.cos(theta), (2/3) * np.sin(theta), 'g:', linewidth=2, label='ROC Boundary (|z| = 2/3)')
            title_str = "X₂(z): Causal Signal (ROC: |z| > 2/3)"

        elif system_choice == "X3(z) = (1 + z^-1 - 2z^-2) / (1 - (13/6)z^-1 + z^-2)":
            # X3: Poles of 1 - 13/6 z^-1 + z^-2 = 0 -> roots at 2 and 0.5 (1/2)
            # Absolutely summable -> ROC includes unit circle -> Annular ROC: 0.5 < |z| < 2
            zeros_roots = np.roots([1, 1, -2]) # numerator: z^2 + z - 2 = 0 -> z = 1, z = -2
            zeros_x, zeros_y = zeros_roots.real, zeros_roots.imag
            poles_x, poles_y = [0.5, 2.0], [0, 0]
            
            roc_mask = (Z_dist > 0.5) & (Z_dist < 2.0)
            ax_pz.imshow(roc_mask, extent=(-3.5, 3.5, -3.5, 3.5), origin='lower', cmap='Greens', alpha=0.25, zorder=0)
            ax_pz.plot(0.5 * np.cos(theta), 0.5 * np.sin(theta), 'g:', linewidth=2, label='Inner ROC (|z| = 0.5)')
            ax_pz.plot(2.0 * np.cos(theta), 2.0 * np.sin(theta), 'g:', linewidth=2, label='Outer ROC (|z| = 2)')
            title_str = "X₃(z): Absolutely Summable (ROC: 0.5 < |z| < 2)"

        # Plot Zeros and Poles
        ax_pz.scatter(zeros_x, zeros_y, s=120, facecolors='none', edgecolors='b', linewidths=2, marker='o', label='Zeros')
        ax_pz.scatter(poles_x, poles_y, s=140, color='purple', marker='x', linewidths=3, label='Poles')

        ax_pz.set_title(title_str, fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)

        # Legend handles in a single straight line
        unit_circle_handle = plt.Line2D([0], [0], color='k', linestyle='--', alpha=0.5, label='Unit Circle')
        pole_handle = plt.Line2D([0], [0], marker='x', color='purple', markersize=8, markeredgewidth=3, linestyle='None', label='Poles')
        zero_handle = plt.Line2D([0], [0], marker='o', markerfacecolor='none', markeredgecolor='b', markersize=8, markeredgewidth=2, linestyle='None', label='Zeros')
        
        ax_pz.legend(handles=[unit_circle_handle, pole_handle, zero_handle], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)

        plt.show()

        # Explanatory comments for Springer submission
        print("-" * 115)
        print("GEOMETRIC & ROC INTERPRETATION:")
        print("1. Poles dictate the boundary limits of the Region of Convergence (ROC).")
        print("2. For causal systems (X1, X2), the ROC is the exterior of a circle defined by the magnitude of the outermost pole.")
        print("3. For absolutely summable systems (X3), stability requires the ROC to include the unit circle (|z| = 1).")
        print("-" * 115)

# Dropdown menu widget for selecting Z-transforms
dropdown = widgets.Dropdown(
    options=[
        "X1(z) = (1 - 0.5z^-1) / (1 + 2z^-1)",
        "X2(z) = [1 - (1/3)z^-1] / [(1 + 0.5z^-1)(1 - (2/3)z^-1)]",
        "X3(z) = (1 + z^-1 - 2z^-2) / (1 - (13/6)z^-1 + z^-2)"
    ],
    value="X1(z) = (1 - 0.5z^-1) / (1 + 2z^-1)",
    description='System:',
    style={'description_width': 'initial'}
)

plot_z_transforms(dropdown.value)

interactive_plot = widgets.interactive(plot_z_transforms, system_choice=dropdown)
display(widgets.VBox([interactive_plot, out]))

HTML(value='\n<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee…